# Kronos - for forecasting 
* created by amazon
* based on transformers architecture
*  a variable should be TIMESTAMP

In [26]:
! pip install "chronos-forecasting>=2.0"

#### Using store_sales.csv

In [27]:
import pandas as pd
import numpy as np
from chronos import Chronos2Pipeline
from sklearn.metrics import root_mean_squared_error
import os
os.chdir('D:\Machine Learning\Datasets')
import warnings
warnings.filterwarnings('ignore')

In [28]:
pipline = Chronos2Pipeline.from_pretrained("amazon/chronos-2")


In [29]:
df = pd.read_csv("store_sales.csv")
df = df.drop(['holiday', 'promo'], axis=1)
df = df.rename(columns={"date":"timestamp","store_id":"store","sales_amount":"sales"})
# df['timestamp'] = pd.date_range(start='1962-01-01', periods=len(df), freq='M')
# df['id'] = 0 # Should be identical
# df.head()

In [30]:
df_1 = df[df['store'] == 1].copy()
train = df_1.iloc[:-12]
test = df_1.iloc[-12:]
train.shape, test.shape

((718, 3), (12, 3))

In [32]:
future_df = test.copy()
future_df['sales'] = None
# future_df['timestamp'] = pd.date_range(start='20-01-01', periods=12, freq='ME')
pred_df = pipline.predict_df(
    train, future_df=future_df,
    validate_inputs=False, #
    prediction_length=12,
    id_column="store",
    timestamp_column="timestamp",
    target='sales',
)
pred_df.columns
y_pred = pred_df["predictions"]

In [33]:
root_mean_squared_error(test['sales'], y_pred)

17.796664340317836

In [36]:
for st in df['store'].unique():
    df_1 = df[df['store'] == st].copy()
    train = df_1.iloc[:-12]
    test = df_1.iloc[-12:]
    train.shape, test.shape
    future_df = test.copy()
    future_df['sales'] = None
# future_df['timestamp'] = pd.date_range(start='20-01-01', periods=12, freq='ME')
    pred_df = pipline.predict_df(
        train, future_df=future_df,
        validate_inputs=False, #
        prediction_length=12,
        id_column="store",
        timestamp_column="timestamp",
        target='sales',
    )
    pred_df.columns
    y_pred = pred_df["predictions"]
    print(f"The RMSE for store {st} =",root_mean_squared_error(test['sales'], y_pred))


The RMSE for store 1 = 17.796664340317836
The RMSE for store 2 = 25.58650385659263
The RMSE for store 3 = 16.33097174478992
The RMSE for store 4 = 15.2480596804013
The RMSE for store 5 = 12.415330142237883
The RMSE for store 6 = 11.458813094922906
The RMSE for store 7 = 14.391783714054663
The RMSE for store 8 = 19.139906086533053
The RMSE for store 9 = 21.69470893632941
The RMSE for store 10 = 13.415704438829192
